In [5]:
import pandas as pd
df = pd.read_csv(r'C:\Users\favaz\Downloads\Employee_Attendance.csv')

In [6]:
print(df)

     Employee_ID     Employee_Name  Department            Designation  \
0        EMP2001       Sneha Patel          IT       Python Developer   
1        EMP2002        Neha Patel          HR             HR Manager   
2        EMP2003        Vishnu Das  Operations   Operations Executive   
3        EMP2004    Nikhil Sharma      Finance        Finance Analyst   
4        EMP2005        Meera Iyer          IT      Software Engineer   
...          ...               ...         ...                    ...   
1525     EMP3321      Ananya Patel         NaN   Operations Executive   
1526     EMP2256         Pooja Das   Marketing      Content Executive   
1527     EMP2455         Aisha Das          IT       Python Developer   
1528     EMP2131      Ishaan Patel          IT       Python Developer   
1529     EMP2428       Karthik Das   Logistics  Logistics Coordinator   

        Location Employment_Type         Joining_Date      Attendance_Date  \
0     Trivandrum       Full Time  2022-12-12 

In [7]:
# 1. Quick overview of columns, data types, and non-null counts
print("--- Dataset Info ---")
print(df.info())

print("\n--- Missing Values Count ---")
print(df.isnull().sum())

print("\n--- Attendance Status Variations ---")
print(df['Attendance_Status'].value_counts(dropna=False))

print("\n--- Duplicate Rows ---")
print(f"Total duplicates: {df.duplicated().sum()}")

--- Dataset Info ---
<class 'pandas.DataFrame'>
RangeIndex: 1530 entries, 0 to 1529
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Employee_ID        1517 non-null   str  
 1   Employee_Name      1530 non-null   str  
 2   Department         1513 non-null   str  
 3   Designation        1513 non-null   str  
 4   Location           1517 non-null   str  
 5   Employment_Type    1530 non-null   str  
 6   Joining_Date       1512 non-null   str  
 7   Attendance_Date    1519 non-null   str  
 8   Attendance_Status  1530 non-null   str  
 9   Late_Minutes       1500 non-null   str  
 10  Overtime_Hours     1523 non-null   str  
 11  Basic_Salary       1507 non-null   str  
 12  Allowance          1527 non-null   str  
 13  Work_Email         1508 non-null   str  
 14  Employee_Status    1525 non-null   str  
dtypes: str(15)
memory usage: 179.4 KB
None

--- Missing Values Count ---
Employee_ID          13
Emp

In [8]:
import pandas as pd
import numpy as np

# Make a working copy
df_clean = df.copy()

# 1. Strip whitespace from all string columns & treat placeholders as NaN
string_cols = df_clean.select_dtypes(include='object').columns
for col in string_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()
    df_clean[col] = df_clean[col].replace({'nan': np.nan, 'None': np.nan, '-': np.nan, '': np.nan})

# 2. Standardize Attendance_Status values
status_map = {
    'work from home': 'WFH',
    'wfh': 'WFH',
    'present': 'Present',
    'absent': 'Absent',
    'half day': 'Half Day',
    'leave': 'Leave'
}
df_clean['Attendance_Status'] = df_clean['Attendance_Status'].str.strip()
df_clean['Attendance_Status'] = df_clean['Attendance_Status'].replace(status_map)

# 3. Parse Dates
df_clean['Joining_Date'] = pd.to_datetime(df_clean['Joining_Date'], errors='coerce')
df_clean['Attendance_Date'] = pd.to_datetime(df_clean['Attendance_Date'], errors='coerce')

# 4. Handle Missing & Inconsistent Values
df_clean['Late_Minutes'] = pd.to_numeric(df_clean['Late_Minutes'], errors='coerce').fillna(0).astype(int)
df_clean['Overtime_Hours'] = pd.to_numeric(df_clean['Overtime_Hours'], errors='coerce').fillna(0).astype(int)
df_clean['Basic_Salary'] = pd.to_numeric(df_clean['Basic_Salary'], errors='coerce')
df_clean['Allowance'] = pd.to_numeric(df_clean['Allowance'], errors='coerce')

# Standardize Department & Designation names
dept_std = {
    'IT ': 'IT', 'it': 'IT', 'I.T.': 'IT',
    'finanace': 'Finance', 'Finance ': 'Finance',
    'Human Resources': 'HR'
}
df_clean['Department'] = df_clean['Department'].replace(dept_std)

desig_std = {
    'DATA ANALYST': 'Data Analyst',
    'DATA ANALYST ': 'Data Analyst',
    'Data analyst': 'Data Analyst',
    'Python developer': 'Python Developer',
    'HR executive': 'HR Executive',
    'Sales executive': 'Sales Executive',
    'Software Engg': 'Software Engineer'
}
df_clean['Designation'] = df_clean['Designation'].replace(desig_std)

# Infer missing Department from known Designation
desig_to_dept = {
    'Customer Care Executive': 'Customer Support',
    'Support Executive': 'Customer Support',
    'Sales Executive': 'Sales',
    'Sales Manager': 'Sales',
    'Business Development Executive': 'Sales',
    'Logistics Coordinator': 'Logistics',
    'Warehouse Executive': 'Logistics',
    'Supply Chain Analyst': 'Logistics',
    'Digital Marketing Specialist': 'Marketing',
    'Content Executive': 'Marketing',
    'Marketing Executive': 'Marketing',
    'Operations Manager': 'Operations',
    'Operations Executive': 'Operations',
    'Process Associate': 'Operations',
    'Senior Accountant': 'Finance',
    'Accountant': 'Finance',
    'Finance Analyst': 'Finance',
    'Recruiter': 'HR',
    'HR Executive': 'HR',
    'HR Manager': 'HR',
    'Software Engineer': 'IT',
    'Python Developer': 'IT',
    'System Administrator': 'IT',
    'Data Analyst': 'IT'
}
df_clean['Department'] = df_clean['Department'].fillna(df_clean['Designation'].map(desig_to_dept))
df_clean['Department'] = df_clean['Department'].fillna('Unknown')

# Impute missing Designation
df_clean['Designation'] = df_clean['Designation'].fillna('Unknown')

# 5. Fix Logical Anomalies
df_clean.loc[df_clean['Attendance_Status'].isin(['Absent', 'Leave']), 'Overtime_Hours'] = 0
df_clean.loc[df_clean['Attendance_Status'].isin(['Absent', 'Leave']), 'Late_Minutes'] = 0
df_clean['Is_Pre_Joining_Record'] = df_clean['Attendance_Date'] < df_clean['Joining_Date']

# 6. Calculate Total Earnings
df_clean['Total_Salary'] = df_clean['Basic_Salary'] + df_clean['Allowance']

# 7. Remove Duplicate Rows
df_clean = df_clean.drop_duplicates()

# Summary of clean dataset
print("Cleaning complete! Shape:", df_clean.shape)
print("\nMissing values after cleaning:\n", df_clean.isnull().sum())
df_clean.head()


Cleaning complete! Shape: (1500, 17)

Missing values after cleaning:
 Employee_ID              13
Employee_Name             0
Department                0
Designation               0
Location                 15
Employment_Type           0
Joining_Date             20
Attendance_Date          15
Attendance_Status         0
Late_Minutes              0
Overtime_Hours            0
Basic_Salary             52
Allowance                25
Work_Email               26
Employee_Status           5
Is_Pre_Joining_Record     0
Total_Salary             76
dtype: int64


C:\Users\favaz\AppData\Local\Temp\ipykernel_4376\853638139.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df_clean.select_dtypes(include='object').columns


,Employee_ID,Employee_Name,Department,Designation,Location,Employment_Type,Joining_Date,Attendance_Date,Attendance_Status,Late_Minutes,Overtime_Hours,Basic_Salary,Allowance,Work_Email,Employee_Status,Is_Pre_Joining_Record,Total_Salary
0,EMP2001,Sneha Patel,IT,Python Developer,Trivandrum,Full Time,2022-12-12,2025-03-27,Present,11,4,51208.0,11909.0,nikhil.0@companymail.com,Active,False,63117.0
1,EMP2002,Neha Patel,HR,HR Manager,Bengaluru,Full Time,2025-02-26,2025-01-09,Half Day,10,0,62248.0,11867.0,aisha.1@companymail.com,Active,True,74115.0
2,EMP2003,Vishnu Das,Operations,Operations Executive,Kochi,Contract,2024-07-13,2025-09-15,WFH,10,6,55645.0,12153.0,aditya.2@companymail.com,Active,False,67798.0
3,EMP2004,Nikhil Sharma,Finance,Finance Analyst,Trivandrum,Part Time,2025-06-22,2025-09-10,Leave,0,0,64099.0,12488.0,aisha.3@companymail.com,Active,False,76587.0
4,EMP2005,Meera Iyer,IT,Software Engineer,Bengaluru,Part Time,2025-04-18,2025-01-10,Present,7,0,39343.0,8684.0,kavya.4@companymail.com,Active,True,48027.0


In [9]:
# 1. View rows where Employee_ID is currently missing (to see before & after)
missing_mask = df_clean['Employee_ID'].isna()
print(f"Number of missing Employee_IDs: {missing_mask.sum()}")

# 2. Extract the numeric portion of Employee_ID (e.g., 'EMP2001' -> 2001.0)
emp_numbers = df_clean['Employee_ID'].str.extract(r'(\d+)')[0].astype(float)

# 3. Linearly interpolate between surrounding rows (e.g., 2025 and 2027 -> 2026)
emp_numbers_interpolated = emp_numbers.interpolate(method='linear').round()

# 4. Format back into 'EMPxxxx' (e.g., 2026 -> 'EMP2026')
filled_ids = emp_numbers_interpolated.apply(lambda x: f"EMP{int(x):04d}" if pd.notnull(x) else np.nan)

# 5. Fill the missing Employee_IDs
df_clean['Employee_ID'] = df_clean['Employee_ID'].fillna(filled_ids)

# 6. (Fallback) If any IDs remain at the very edges, match by Employee_Name
name_to_id = df_clean.dropna(subset=['Employee_ID']).groupby('Employee_Name')['Employee_ID'].first()
df_clean['Employee_ID'] = df_clean['Employee_ID'].fillna(df_clean['Employee_Name'].map(name_to_id))

# Verify that missing values are resolved
print(f"Missing Employee_IDs remaining: {df_clean['Employee_ID'].isna().sum()}")

Number of missing Employee_IDs: 13
Missing Employee_IDs remaining: 0


In [10]:
# Export cleaned dataset
df_clean.to_csv("cleaned_data.csv", index=False)
print("Successfully saved cleaned dataset to cleaned_data.csv!")


Successfully saved cleaned dataset to cleaned_data.csv!
